# Day 13 — Delta Lake Change Data Feed (CDF) & Incremental Processing

- if CDF NOT ENABLED, then you have to **compare the full table** to identify the changes
- if CDF ENABLED --> Delta Lake exposes the **row-level changes** associated with table commits.

CDF allows downstream consumers to** process changes instead of repeatedly comparing entire table** snapshots.

**NOTE**- 
- **Checkpoint**: Tracks streaming processing progress and helps Spark recover after failures.
- **Commit version:** Helps determine whether an incoming change is newer than the change already applied to a customer.

## CDF using SQL
```
    ALTER TABLE workspace.customer360_bronze.customers_cdf_day13
    SET TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    );


    SELECT
    custId,
    customerName,
    status,
    balance,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.customer360_bronze.customers_cdf_day13',
    2
)
ORDER BY _commit_version, custId;
````


## CDF using pyspark

````
    source_table = "workspace.customer360_bronze.customers_cdf_day13"

    dfCDFStream = (
        spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", 2)
        .table(source_table)
    )

    target_table = "workspace.customer360_silver.customers_cdf_history_day13"

checkpoint_path = (
    "/Volumes/workspace/customer360_bronze/"
    "ingestion_metadata/checkpoints/"
    "customers_cdf_history_day13"
)

query = (
    dfCDFStream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
````

## Interview question
## Suppose you run the streaming query three times without adding new source records. 
## Will the Silver history table contain 5, 10, or 15 records? Why?
CDF identifies changed rows; checkpointing prevents reprocessing previously committed source progress during normal recovery. With a Delta sink, this supports exactly-once processing under the supported guarantees.

Important interview concept
Why do we filter out update_preimage?

Because update_preimage contains the old values before an UPDATE. For an SCD Type 1 current-state table, we want the latest values from update_postimage.

Also, notice that our MERGE handles three scenarios:
- MATCHED + DELETE: Removes an existing customer.
- MATCHED + INSERT/UPDATE: Updates an existing customer.
- NOT MATCHED + INSERT/UPDATE: Inserts a new customer.
A DELETE for a customer that does not exist in the target is ignored.


````

%sql
dfLatestChanges.createOrReplaceTempView("latest_customer_changes")


MERGE INTO workspace.customer360_silver.customers_current_day13 AS target
USING latest_customer_changes AS source
ON target.custId = source.custId

WHEN MATCHED AND source._change_type = 'delete'
THEN DELETE

WHEN MATCHED AND source._change_type IN ('insert', 'update_postimage')
THEN UPDATE SET
    target.customerName = source.customerName,
    target.status = source.status,
    target.balance = source.balance

WHEN NOT MATCHED AND source._change_type IN ('insert', 'update_postimage')
THEN INSERT (
    custId,
    customerName,
    status,
    balance
)
VALUES (
    source.custId,
    source.customerName,
    source.status,
    source.balance
);
````

%md
**Q1. What is the difference between Delta Change Data Feed and CDC?**

CDF is a Delta Lake feature that exposes row-level changes made to a Delta table. CDC is a broader approach for capturing and propagating changes between systems. CDF can serve as a source for a CDC pipeline.

**Q2. How would you build an incremental pipeline using CDF?**

I would enable CDF on the source Delta table, read changes using Structured Streaming, and maintain a checkpoint to track processing progress. For a current-state Silver table, I would apply inserts, updates, and deletes using Delta MERGE, with appropriate change ordering and idempotent retry handling.

**Q3. What is the difference between CDF and SCD Type 2?**

CDF provides a feed of changes from a Delta table. SCD Type 2 is a data-modeling technique that preserves historical versions of business records. CDF can be used to drive an SCD Type 2 pipeline, but it does not automatically create SCD Type 2 history.

**Why did we use checkpoints?**
We streamed CDF changes from Bronze into a Silver history table using:
````
query = (
    dfCDFStream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
    
````
Your Silver history table initially contained four changes. After inserting customer 1005, it contained five.

Running the stream three more times with the same checkpoint and no new source changes kept the total at five records.

CDF tells us what changed. Checkpointing tells Spark what has already been processed.

### 1. What is Delta Change Data Feed?
CDF captures row-level changes made to a Delta table after CDF is enabled.

Instead of comparing the entire source and target tables, we can read only the changed records and apply them downstream.

In our Customer360 project, we enabled CDF using:
````
ALTER TABLE workspace.customer360_bronze.customers_cdf_day13
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);
````

Four CDF change types
````
Change type	        Meaning	                Use in SCD Type 1
insert	            New record	            Insert / upsert
update_preimage	    Values before UPDATE	Ignore
update_postimage	Values after UPDATE	    Update / upsert
delete	            Deleted record	        Delete
````

Remember: One UPDATE typically generates two CDF records: update_preimage and update_postimage.

Consider this scenario:
1. Spark reads CDF changes from versions 10–15.
2. foreachBatch executes a MERGE into the Silver table.
3. The MERGE succeeds.
4. The job crashes before Spark records successful batch completion in its checkpoint.
5. Spark restarts and retries the same batch.
The checkpoint correctly tells Spark that the batch wasn't marked complete. But the target table may already contain its changes.

**This is why the MERGE must be idempotent.
Interview-ready answer**

I would use Structured Streaming checkpoints to track processing progress and recover from failures. Since foreachBatch can retry a micro-batch after a failure, I would make the Delta MERGE idempotent. I would also track the last applied change version to prevent duplicate or older changes from overwriting newer data. This ensures the target remains consistent even when a batch is retried.

## Quick follow-up question
Suppose customer 1001 has a balance of $500.

A CDF update changes the balance to $650, and the MERGE succeeds. The job crashes before checkpoint completion.

When the same batch is retried, what happens if our MERGE uses:
````
UPDATE SET target.balance = source.balance
````

versus:
````
UPDATE SET target.balance = target.balance + source.balance
````

**Which approach is idempotent, and why?**

Correct! You've understood the key concept of idempotency.

Your explanation of the first statement is exactly right. There's one small correction to your calculation for the second statement.
Let's walk through the failure scenario
Initial balance: $500
CDF updated balance: $650

The first MERGE succeeds, but Spark crashes before recording successful batch completion in the checkpoint.
```
Stage	            target.balance = source.balance	target.balance = target.balance + source.balance
Initial target	    $500	                        $500
First execution	    $650	                        $1,150
Retry of same batch	$650	                        $1,800
Idempotent?	        Yes	                            No
````

Why?
**Statement 1 — Idempoten--> UPDATE SET target.balance = source.balance**
Every execution assigns the same value:
- First execution: $650
- Retry: $650
The result doesn't change when the same operation is repeated.

**Statement 2 — Not idempotent**
UPDATE SET target.balance = target.balance + source.balance


Every execution adds another $650:
- First execution: $500 + $650 = $1,150
- Retry: $1,150 + $650 = $1,800
This causes incorrect balances.

One important distinction: we're assuming the CDF record contains the new absolute balance, not a payment amount or balance adjustment.

**Senior-level interview takeaway**
````
Checkpointing helps Spark recover its processing progress, while idempotent MERGE logic ensures that retrying the same batch doesn't change an already-correct target result.
````

Also remember that a simple assignment is idempotent for replaying the same event, but production pipelines still need change-version checks to prevent an older event from overwriting a newer value.

-->
Suppose the target table contains:
custId	balance	lastCommitVersion
1001	850	15

An older CDF event arrives:
custId	balance	_commit_version
1001	650	12
**
Should the MERGE update the balance to $650 or retain $850? How would you prevent the older change from overwriting the newer one?**


**Implement using merge**
````
MERGE INTO customers_current AS target
USING latest_customer_changes AS source
ON target.custId = source.custId

WHEN MATCHED
  AND source._commit_version > target.lastCommitVersion
THEN UPDATE SET
  target.balance = source.balance,
  target.lastCommitVersion = source._commit_version;
  ````
s**ource._commit_version > target.lastCommitVersion  -->   prevents older changes from overwriting newer values. This assumes both versions come from the same source Delta table and that each customer's changes are ordered correctly.**


